# Grid Stability Prediction: Baselines, Generalization, CCT Speedup & SHAP XAI
### EE Minor Project (7th Semester) — National Institute of Technology (NIT) Raipur
**Authors:** Sanskar Gupta (23117091), Daksh Tripathi (23117036), Karandeep Taram (23117052), Veena Gupta (23117112)

---

### Research Summary
Transient Stability Assessment (TSA) and Critical Clearing Time (CCT) determination traditionally require solving stiff differential-algebraic equations (DAEs) using numerical Time-Domain Simulation (TDS), which takes tens of seconds per disturbance scenario.

This notebook demonstrates:
1. **High in-distribution accuracy** (~100%) on standard stratified 80/20 train/test splits.
2. **The deceptive nature of random splits**: showing severe generalization collapse when testing on unseen buses (15.2% missed blackouts) or unseen heavy loads (accuracy drops to 63.9%).
3. **Physics-informed invariant features** ($v_{\text{sag\_impulse}} = (1 - v_{\min}) \times t_{\text{clear}}$ from the Equal Area Criterion) cutting topological blackout error by more than half.
4. **Direct CCT Regression achieving >33,000x speedup** (0.7 ms vs 23.9 s) with an RMSE of 25 ms.
5. **TreeSHAP explainability** proving that ML decision boundaries align with the Swing Equation.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure project root is in sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.models.data_pipeline import load_and_clean_data, get_standard_split
from src.models.train_baselines import train_and_evaluate_baselines
from src.models.generalization_experiments import run_generalization_experiments
from src.models.train_cct_regressor import train_cct_regressors
from src.explain.shap_analysis import run_shap_analysis

print("Environment initialized successfully.")

## 1. Load Dataset & Inspection
We load the 450 simulated transient disturbance scenarios on the IEEE 39-bus New England system generated via ANDES.

In [ ]:
df = load_and_clean_data('../data/processed/dataset.csv')
print(f"Total Scenarios: {len(df)}")
print(f"Class Distribution: {df['stable'].value_counts().to_dict()} (1=Stable, 0=Unstable)")
df.head()

## 2. In-Distribution Baseline Training (Standard 80/20 Random Split)
We evaluate Logistic Regression, Random Forest, XGBoost, and a 2-layer Neural Network (MLP) with 5-fold cross-validation.

In [ ]:
baseline_results, trained_models = train_and_evaluate_baselines()
results_table = pd.DataFrame(baseline_results).T[['accuracy', 'macro_f1', 'unstable_recall', 'missed_instability_rate', 'roc_auc']]
display(results_table)

In [ ]:
print("Baseline Confusion Matrices:")
display(Image('../reports/figures/baseline_confusion_matrices.png'))

print("Baseline ROC Curves:")
display(Image('../reports/figures/baseline_roc_curves.png'))

## 3. Core Research Contribution: Held-Out Generalization Experiments
To test true operational transferability, we deliberately hold out:
1. **Unseen Fault Buses** (Buses 29 and 39 — tests spatial/topological transfer)
2. **Unseen Heavy Operating Loads** (110% and 120% stress — tests out-of-distribution extrapolation)
3. **Unseen N-1 Line Outage** (Line 16-17 — tests post-contingency transfer)

We also test **Physics-Informed Feature Engineering** using the Equal Area Criterion acceleration impulse:  
$$v_{\text{sag\_impulse}} = (1.0 - v_{\min}) \times t_{\text{clear}}$$

In [ ]:
summary_df = run_generalization_experiments()
display(summary_df)

## 4. Visualizing the Generalization Gap & Missed Instability Risks

In [ ]:
print("Generalization Gap (In-Distribution vs Out-of-Distribution F1-score):")
display(Image('../reports/figures/generalization_gap.png'))

print("Critical Power System Risk (Missed Instability Rate across Models):")
display(Image('../reports/figures/generalization_missed_instability.png'))

## 5. Phase 5: Critical Clearing Time (CCT) Regression & 33,000x Speedup
Predicting the continuous Critical Clearing Time directly from pre-fault operating conditions avoids running 10+ numerical TDS iterations.
We compare Ridge Regression, Random Forest, and XGBoost against ANDES bisection ground truth.

In [ ]:
cct_results, cct_models = train_cct_regressors('../data/processed/cct_dataset.csv')
cct_table = pd.DataFrame(cct_results).T[['r2_score', 'rmse_sec', 'mae_sec', 'inference_time_ms', 'speedup_factor']]
display(cct_table)

In [ ]:
print("CCT Parity Plot (Predicted vs Actual CCT in seconds):")
display(Image('../reports/figures/cct_parity_plot.png'))

print("Inference Latency Benchmark (ANDES Numerical TDS vs ML Regressors):")
display(Image('../reports/figures/cct_speed_comparison.png'))

## 6. Phase 6: TreeSHAP Explainability & Physical Validation
We compute exact Shapley values using TreeSHAP to verify whether feature attributions reflect physical laws (Swing Equation: $M\ddot{\delta} = P_m - P_e$).

In [ ]:
shap_results = run_shap_analysis()

print("SHAP Global Feature Importance Beeswarm Plot:")
display(Image('../reports/figures/shap_summary_beeswarm.png'))

print("SHAP Local Waterfall Explanations (Stable vs Unstable Scenario):")
display(Image('../reports/figures/shap_waterfall_stable.png'))
display(Image('../reports/figures/shap_waterfall_unstable.png'))

## 7. Phase 7: Interactive Digital Twin Streamlit Dashboard
To interactively explore scenarios with real-time stability classification, continuous CCT safety margins, and live SHAP waterfall decomposition, launch the Streamlit app:

```bash
python -m streamlit run app/streamlit_app.py
```

The web app will open at `http://localhost:8501`.